In [41]:
import pandas as pd
import numpy as np
df = pd.read_csv('african_crises.csv')

In [42]:
df.head()

,case,cc3,country,year,systemic_crisis,exch_usd,domestic_debt_in_default,sovereign_external_debt_default,gdp_weighted_default,inflation_annual_cpi,independence,currency_crises,inflation_crises,banking_crisis
0,1,DZA,Algeria,1870,1,0.052264,0,0,0.0,3.441456,0,0,0,crisis
1,1,DZA,Algeria,1871,0,0.052798,0,0,0.0,14.149140,0,0,0,no_crisis
2,1,DZA,Algeria,1872,0,0.052274,0,0,0.0,-3.718593,0,0,0,no_crisis
3,1,DZA,Algeria,1873,0,0.051680,0,0,0.0,11.203897,0,0,0,no_crisis
4,1,DZA,Algeria,1874,0,0.051308,0,0,0.0,-3.848561,0,0,0,no_crisis


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1059 entries, 0 to 1058
Data columns (total 14 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   case                             1059 non-null   int64  
 1   cc3                              1059 non-null   object 
 2   country                          1059 non-null   object 
 3   year                             1059 non-null   int64  
 4   systemic_crisis                  1059 non-null   int64  
 5   exch_usd                         1059 non-null   float64
 6   domestic_debt_in_default         1059 non-null   int64  
 7   sovereign_external_debt_default  1059 non-null   int64  
 8   gdp_weighted_default             1059 non-null   float64
 9   inflation_annual_cpi             1059 non-null   float64
 10  independence                     1059 non-null   int64  
 11  currency_crises                  1059 non-null   int64  
 12  inflation_crises    

In [44]:
# Missing count
missing_count = df.isnull().sum()

# Missing percent
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_table = pd.DataFrame({
    'missing_count': missing_count,
    'missing_percent': missing_percent
})
missing_table

,missing_count,missing_percent
case,0,0.0
cc3,0,0.0
country,0,0.0
year,0,0.0
systemic_crisis,0,0.0
exch_usd,0,0.0
domestic_debt_in_default,0,0.0
sovereign_external_debt_default,0,0.0
gdp_weighted_default,0,0.0
inflation_annual_cpi,0,0.0


In [45]:
# date
df["year"] = df["year"].astype(int)

# bool conversions
binary_map = {0:"No",1:"Yes"}

df["systemic_crisis"] = df["systemic_crisis"].map(binary_map)
df["currency_crisis"] = df["currency_crises"].map(binary_map)
df["inflation_crisis"] = df["inflation_crises"].map(binary_map)

df["domestic_debt_default"] = df["domestic_debt_in_default"].map(binary_map)
df["external_debt_default"] = df["sovereign_external_debt_default"].map(binary_map)

# banking crisis clean
df["banking_crisis"] = df["banking_crisis"].replace({
"crisis":"Yes",
"no_crisis":"No"
})

# SQL compatible types
df["case"] = df["case"].astype(int)
df["exch_usd"] = df["exch_usd"].astype(float)
df["inflation_annual_cpi"] = df["inflation_annual_cpi"].astype(float)

In [46]:
# Rename
df = df.rename(columns={

"cc3":"iso_country_code",
   
"case":"country_id",

"exch_usd":"usd_exchange_rate",

"inflation_annual_cpi":"annual_inflation_rate",

"gdp_weighted_default":"debt_default_impact_gdp",

"systemic_crisis":"systemic_financial_crisis",

"domestic_debt_default":"domestic_debt_default_event",

"external_debt_default":"external_debt_default_event"

})

In [47]:
## create of columns
# inflation risk level
def classify_inflation(x):
    
    if x < 5:
        return "Low inflation"
    
    elif x < 15:
        return "Moderate inflation"
    
    else:
        return "High inflation"

df["inflation_risk_level"] = df["annual_inflation_rate"].apply(classify_inflation)

# Total crisis events
binary_map = {"Yes":1,"No":0}

df["total_crisis_events"] = (
    df["banking_crisis"].map(binary_map) +
    df["currency_crisis"].map(binary_map) +
    df["inflation_crisis"].map(binary_map) +
    df["domestic_debt_default_event"].map(binary_map) +
    df["external_debt_default_event"].map(binary_map)
)

# economic crisis severity
def crisis_severity(x):
    
    if x == 0:
        return "Stable economy"
    
    elif x == 1:
        return "Minor crisis"
    
    elif x <= 3:
        return "Moderate crisis"
    
    else:
        return "Severe crisis"

df["economic_crisis_severity"] = df["total_crisis_events"].apply(crisis_severity)

# Country maturity level
df["years_since_independence"] = df["year"] - df["independence"]

def country_maturity(age):
    
    if age < 30:
        return "Young nation"
    
    elif age < 60:
        return "Developing nation"
    
    else:
        return "Established nation"

df["country_maturity_level"] = df["years_since_independence"].apply(country_maturity)

# Crisis period
def crisis_period(year):
    
    if year < 1980:
        return "Early period"
    
    elif year <= 2000:
        return "Reform period"
    
    else:
        return "Modern period"

df["crisis_period"] = df["year"].apply(crisis_period)

In [48]:
df.drop(columns=[
"country_id",
"iso_country_code",
"years_since_independence",
"currency_crises",
"sovereign_external_debt_default",
"domestic_debt_in_default",
"inflation_crises",
"independence"
], inplace = True)    

In [49]:
df.head()

,country,year,systemic_financial_crisis,usd_exchange_rate,debt_default_impact_gdp,annual_inflation_rate,banking_crisis,currency_crisis,inflation_crisis,domestic_debt_default_event,external_debt_default_event,inflation_risk_level,total_crisis_events,economic_crisis_severity,country_maturity_level,crisis_period
0,Algeria,1870,Yes,0.052264,0.0,3.441456,Yes,No,No,No,No,Low inflation,1.0,Minor crisis,Established nation,Early period
1,Algeria,1871,No,0.052798,0.0,14.149140,No,No,No,No,No,Moderate inflation,0.0,Stable economy,Established nation,Early period
2,Algeria,1872,No,0.052274,0.0,-3.718593,No,No,No,No,No,Low inflation,0.0,Stable economy,Established nation,Early period
3,Algeria,1873,No,0.051680,0.0,11.203897,No,No,No,No,No,Moderate inflation,0.0,Stable economy,Established nation,Early period
4,Algeria,1874,No,0.051308,0.0,-3.848561,No,No,No,No,No,Low inflation,0.0,Stable economy,Established nation,Early period


In [50]:
df.to_excel("african_crises.xlsx",index=False,sheet_name="african_crises")

In [53]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

user = "root"
password = "mysql2026"
host = "localhost"
port = 3306
db = "esmel"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}",
    echo=False
)

In [54]:
engine.connect()

In [55]:
df.to_sql(
    name="african_crises",
    con=engine,
    if_exists="replace",
    index=False
)

1059